# Week 8 (W8-1) - Supplier reliability: delivery, lead-time variation and concentration
**Owner:** Chandrashekhar Deurghare (BI Analyst)

**Question.** Which of the 8 suppliers deliver late or vary most in lead time, how concentrated is purchasing on any one of them, and are any differences larger than chance would produce?

**Basis.** The unit is the **purchase order** (24,000 POs; 21,630 received, 2,370 cancelled). A PO is **on time** if it was received on or before its expected delivery date (`delivery_delay_days <= 0`). **Lead time** is received date minus order date. Money is `total_cost` before GST on received POs. Intervals are 95% bootstrap intervals over POs (5,000 resamples, seed 42); group tests are Holm-adjusted.

**Limits.** The data has no promised-versus-quoted price, no quality or defect field and no cost of a late delivery, so reliability here means timing and cancellation only. Purchase spend is about 14 times net sales (Q10 unresolved), so **money figures are descriptive and carry the label `PROCUREMENT SCALE - purchase spend is about 14x sales revenue (Q10 unresolved); money figures are descriptive`**. Rates, shares and lead times do not depend on that scale.

In [1]:
import pandas as pd, numpy as np, os
from pathlib import Path
from scipy import stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
SURF,INK,INK2,GRID,BLUE,ORANGE,AQUA,MUTED="#fcfcfb","#0b0b0b","#52514e","#e4e3df","#2a78d6","#eb6834","#1baf7a","#b9b8b2"
SEED=42; NB=5000
def find_dir(env_var, marker, rels):
    # 1) environment variable, 2) known layouts from here upward, 3) bounded search of the parent folders
    if os.environ.get(env_var) and (Path(os.environ[env_var]) / marker).exists():
        return Path(os.environ[env_var])
    bases = [Path.cwd(), *Path.cwd().parents]
    for base in bases:
        for rel in rels:
            if (base / rel / marker).exists():
                return base / rel
    skip = {".git", "node_modules", "__pycache__", ".ipynb_checkpoints", ".venv", "venv", "site-packages"}
    for base in bases[:4]:
        found = []
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in skip]
            if len(Path(root).parts) - len(base.parts) >= 7:
                dirs[:] = []
            if marker in files:
                found.append(Path(root))
        if found:
            found.sort(key=lambda p: (p.name != Path(rels[0]).name, len(p.parts), str(p)))
            return found[0]
    return None
MERGED = find_dir("CADETX_MERGED", "sales_orders_full.csv", ["merged", "data/merged", "week-02/reports/week_02/merged", "reports/week_02/merged", "week-02/reports/merged", "reports/merged"])
assert MERGED is not None, "Could not find the Week 2 merged folder. Run from inside the repository, or set CADETX_MERGED to the folder."
print("merged data folder found")
checks=[]
def check(name,expected,actual,ok):
    checks.append(dict(Check_Item=name,Expected=expected,Actual=actual,Status="PASS" if ok else "FAIL")); assert ok,f"{name}: expected {expected}, got {actual}"
def style(ax):
    ax.set_facecolor(SURF); ax.figure.set_facecolor(SURF)
    for s in ["top","right"]: ax.spines[s].set_visible(False)
    for s in ["left","bottom"]: ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK2); ax.yaxis.grid(True,color=GRID,lw=.8); ax.set_axisbelow(True)
def holm(p):
    p=np.asarray(p,float); o=np.argsort(p); m=len(p); adj=np.empty(m); run=0
    for k,i in enumerate(o): run=max(run,(m-k)*p[i]); adj[i]=min(1,run)
    return adj

OUT=Path("w81_outputs"); OUT.mkdir(exist_ok=True); CH=Path("w81_charts"); CH.mkdir(exist_ok=True)
LABEL="PROCUREMENT SCALE - purchase spend is about 14x sales revenue (Q10 unresolved); money figures are descriptive"
rng=np.random.default_rng(SEED)
cols=["po_id","line_number","product_id","line_total","supplier_id","supplier_name","supplier_type","supplier_region","supplier_lead_days","reliability_score","order_date","expected_delivery_date","received_date","po_status","total_cost","delivery_delay_days","lead_time_actual_days","criticality_level"]
L=pd.read_csv(MERGED/"purchase_orders_full.csv",low_memory=False,usecols=cols,parse_dates=["order_date","expected_delivery_date","received_date"])

merged data folder found


## 1. Load and reconcile

In [2]:
check("PO lines",155495,len(L),len(L)==155495)
H=L.drop_duplicates("po_id").copy()
check("purchase orders",24000,len(H),len(H)==24000)
var=int((L.groupby("po_id")[["supplier_id","total_cost","po_status","order_date","expected_delivery_date"]].nunique()>1).sum().sum()); check("header fields are constant within each PO (supplier, cost, status, dates)",0,var,var==0)
check("eight suppliers",8,H.supplier_id.nunique(),H.supplier_id.nunique()==8)
R=H[H.po_status=="Received"].copy(); X=H[H.po_status=="Cancelled"].copy()
check("received POs",21630,len(R),len(R)==21630); check("cancelled POs",2370,len(X),len(X)==2370)
check("cancelled POs have no received date",0,int(X.received_date.notna().sum()),int(X.received_date.notna().sum())==0)
d1=int(((R.received_date-R.expected_delivery_date).dt.days!=R.delivery_delay_days).sum()); check("delay = received date - expected date on every received PO",0,d1,d1==0)
d2=int(((R.received_date-R.order_date).dt.days!=R.lead_time_actual_days).sum()); check("lead time = received date - order date on every received PO",0,d2,d2==0)
check("received spend before GST (INR B)",312.03,round(R.total_cost.sum()/1e9,2),abs(R.total_cost.sum()/1e9-312.03)<0.005)
R["on_time"]=(R.delivery_delay_days<=0).astype(int); R["late"]=(R.delivery_delay_days>0).astype(int); R["late4"]=(R.delivery_delay_days>=4).astype(int)
R["gap"]=R.lead_time_actual_days-R.supplier_lead_days
S=np.array(sorted(R.supplier_id.unique())); NAME=H.drop_duplicates("supplier_id").set_index("supplier_id").supplier_name
print("on-time overall %.1f%%, late %.1f%%, early %.1f%%"%(R.on_time.mean()*100,R.late.mean()*100,(R.delivery_delay_days<0).mean()*100))

on-time overall 84.9%, late 15.1%, early 4.5%


## 2. Supplier scorecard with intervals

In [3]:
def boot_mean(v,rng):
    v=np.asarray(v,float); ix=rng.integers(0,len(v),(NB,len(v))); m=v[ix].mean(axis=1); return np.percentile(m,[2.5,97.5])
rows=[]
for s in S:
    r=R[R.supplier_id==s]; h=H[H.supplier_id==s]
    ot=boot_mean(r.on_time,rng); dl=boot_mean(r.delivery_delay_days,rng); cn=boot_mean((h.po_status=="Cancelled").astype(int),rng)
    rows.append(dict(supplier_id=s,supplier_name=NAME[s],supplier_type=h.supplier_type.iloc[0],region=h.supplier_region.iloc[0],pos=len(h),received=len(r),
      cancel_rate=(h.po_status=="Cancelled").mean()*100,cancel_lo=cn[0]*100,cancel_hi=cn[1]*100,
      on_time_rate=r.on_time.mean()*100,on_time_lo=ot[0]*100,on_time_hi=ot[1]*100,late_rate=r.late.mean()*100,late_4plus_rate=r.late4.mean()*100,
      mean_delay_days=r.delivery_delay_days.mean(),delay_lo=dl[0],delay_hi=dl[1],p95_delay_days=r.delivery_delay_days.quantile(.95),
      contract_lead_days=r.supplier_lead_days.iloc[0],lead_mean=r.lead_time_actual_days.mean(),lead_sd=r.lead_time_actual_days.std(),lead_p5=r.lead_time_actual_days.quantile(.05),lead_p95=r.lead_time_actual_days.quantile(.95),
      lead_vs_contract=r.gap.mean(),spend_inr=r.total_cost.sum(),spend_share=r.total_cost.sum()/R.total_cost.sum()*100,reliability_score=h.reliability_score.iloc[0]))
SC=pd.DataFrame(rows); SC.to_csv(OUT/"w81_supplier_scorecard.csv",index=False)
# independent recomputation with plain loops for three suppliers
for s in S[[0,3,6]]:
    n=ok=0
    for a,b in zip(R[R.supplier_id==s].received_date,R[R.supplier_id==s].expected_delivery_date):
        n+=1; ok+= (a<=b)
    got=SC.loc[SC.supplier_id==s,"on_time_rate"].iloc[0]; check(f"{s} on-time rate recomputed with a plain loop",round(ok/n*100,3),round(got,3),abs(ok/n*100-got)<1e-9)
check("PO counts add up to 24,000",24000,int(SC.pos.sum()),int(SC.pos.sum())==24000)
check("spend shares add up to 100%",100.0,round(SC.spend_share.sum(),6),abs(SC.spend_share.sum()-100)<1e-6)
print(SC[["supplier_name","on_time_rate","on_time_lo","on_time_hi","mean_delay_days","cancel_rate","lead_mean","lead_sd","contract_lead_days","spend_share"]].round(2).to_string(index=False))

                  supplier_name  on_time_rate  on_time_lo  on_time_hi  mean_delay_days  cancel_rate  lead_mean  lead_sd  contract_lead_days  spend_share
          Shenzhen OEM Supplies         84.92       83.53       86.28             0.50         9.88      19.50     1.72                  19        12.64
Guangzhou Local Vendor Supplies         84.97       83.57       86.34             0.52        10.28      10.52     1.72                  10        12.09
  Shanghai Distributor Supplies         85.02       83.69       86.30             0.53        10.22      16.53     1.78                  16        12.56
    Ningbo Distributor Supplies         84.83       83.48       86.26             0.50        10.18      18.50     1.69                  18        12.19
           Tianjin OEM Supplies         85.33       83.99       86.67             0.49         9.53      28.49     1.65                  28        12.90
   Qingdao Distributor Supplies         85.21       83.90       86.56             

## 3. Do the suppliers differ more than chance?
Five tests, Holm-adjusted together. Lead time itself differs by design (each supplier has its own contracted lead time), so the lead-time test is on the **gap to contract** and on the **spread** within each supplier.

In [4]:
def cramers(tab): c2=stats.chi2_contingency(tab,correction=False)[0]; return c2/tab.values.sum()**0.5/ (min(tab.shape)-1)**0.5
def eps2(groups): h=stats.kruskal(*groups).statistic; n=sum(len(g) for g in groups); return max(0.0,(h-len(groups)+1)/(n-len(groups)))
T=[]
tab=pd.crosstab(R.supplier_id,R.on_time); p=stats.chi2_contingency(tab,correction=False)[1]; T.append(("on-time rate","chi-square","PO",len(R),p,cramers(tab)))
tab=pd.crosstab(H.supplier_id,H.po_status); p=stats.chi2_contingency(tab,correction=False)[1]; T.append(("PO cancellation rate","chi-square","PO",len(H),p,cramers(tab)))
g=[R[R.supplier_id==s].delivery_delay_days.values for s in S]; T.append(("delay in days","Kruskal-Wallis","PO",len(R),stats.kruskal(*g)[1],eps2(g)))
g=[R[R.supplier_id==s].gap.values for s in S]; T.append(("lead time minus contract","Kruskal-Wallis","PO",len(R),stats.kruskal(*g)[1],eps2(g)))
g=[R[R.supplier_id==s].lead_time_actual_days.values for s in S]; lv=stats.levene(*[x-np.median(x) for x in g],center="median"); T.append(("spread of lead time","Levene (median-centred)","PO",len(R),lv[1],np.nan))
TT=pd.DataFrame(T,columns=["dimension","test","unit","n","p_value","effect_size"]); TT["p_holm"]=holm(TT.p_value.values); TT.to_csv(OUT/"w81_supplier_tests.csv",index=False)
print(TT.round(4).to_string(index=False))
rs=stats.spearmanr(SC.reliability_score,SC.on_time_rate).statistic
perm=np.array([stats.spearmanr(rng.permutation(SC.reliability_score.values),SC.on_time_rate).statistic for _ in range(NB)]); pr=(np.abs(perm)>=abs(rs)-1e-12).mean()
print("reliability_score vs observed on-time rate: Spearman %.2f, permutation p %.2f (n=8 suppliers)"%(rs,pr))
pd.DataFrame([dict(measure="reliability_score vs on-time rate",n=8,spearman=rs,perm_p=pr)]).to_csv(OUT/"w81_reliability_score_check.csv",index=False)
check("on-time test has 21,630 POs",21630,int(TT.n.iloc[0]),int(TT.n.iloc[0])==21630)
check("on-time rates of the eight suppliers fit within a few points (range < 3 points)","<3",round(SC.on_time_rate.max()-SC.on_time_rate.min(),2),SC.on_time_rate.max()-SC.on_time_rate.min()<3)
check("every Holm-adjusted p is at least the raw p","True","True",bool((TT.p_holm>=TT.p_value-1e-12).all()))

               dimension                    test unit     n  p_value  effect_size  p_holm
            on-time rate              chi-square   PO 21630   0.9702       0.0122     1.0
    PO cancellation rate              chi-square   PO 24000   0.9077       0.0177     1.0
           delay in days          Kruskal-Wallis   PO 21630   0.8767       0.0000     1.0
lead time minus contract          Kruskal-Wallis   PO 21630   0.8767       0.0000     1.0
     spread of lead time Levene (median-centred)   PO 21630   0.8535          NaN     1.0
reliability_score vs observed on-time rate: Spearman -0.44, permutation p 0.30 (n=8 suppliers)


## 4. Lead-time spread: what it means for safety stock

In [5]:
sd_within=R.groupby("supplier_id").lead_time_actual_days.std()
LT=SC[["supplier_id","supplier_name","contract_lead_days","lead_mean","lead_sd","lead_p5","lead_p95","lead_vs_contract","p95_delay_days","late_4plus_rate"]].copy()
LT["cv_pct"]=LT.lead_sd/LT.lead_mean*100; LT.to_csv(OUT/"w81_lead_time_spread.csv",index=False)
print("between-supplier range of mean lead time: %.1f to %.1f days"%(SC.lead_mean.min(),SC.lead_mean.max()))
print("within-supplier SD of lead time: %.2f to %.2f days"%(sd_within.min(),sd_within.max()))
check("between-supplier spread of lead time is more than 5 times the within-supplier SD","True","True",(SC.lead_mean.max()-SC.lead_mean.min())>5*sd_within.max())
check("every supplier is late on average by less than one day",0.0,round(SC.mean_delay_days.max(),2),SC.mean_delay_days.max()<1)
share_late_val=R.loc[R.late==1,"total_cost"].sum()/R.total_cost.sum()*100; print("share of received spend delivered late: %.1f%%  [%s]"%(share_late_val,LABEL))

between-supplier range of mean lead time: 10.5 to 28.5 days
within-supplier SD of lead time: 1.65 to 1.78 days
share of received spend delivered late: 15.3%  [PROCUREMENT SCALE - purchase spend is about 14x sales revenue (Q10 unresolved); money figures are descriptive]


## 5. Is reliability changing over time, or different for critical products?
Year is the order year. A PO is **critical** if any of its lines is a High-criticality product.

In [6]:
R["year"]=R.order_date.dt.year; crit=L.assign(hi=(L.criticality_level=="High")).groupby("po_id").hi.max()
R["critical"]=R.po_id.map(crit).astype(int)
yr=R.groupby(["supplier_id","year"]).on_time.mean().unstack()*100; yr.to_csv(OUT/"w81_on_time_by_year.csv")
tab=pd.crosstab(R.year,R.on_time); p_year=stats.chi2_contingency(tab,correction=False)[1]
mo=R.assign(ym=R.order_date.dt.to_period("M")).groupby("ym").on_time.mean().values; idx=np.arange(len(mo))
rs2=stats.spearmanr(idx,mo).statistic; pm=(np.abs([stats.spearmanr(idx,rng.permutation(mo)).statistic for _ in range(NB)])>=abs(rs2)-1e-12).mean()
tab2=pd.crosstab(R.critical,R.on_time); p_crit=stats.chi2_contingency(tab2,correction=False)[1]
sup_crit=[stats.chi2_contingency(pd.crosstab(R[R.supplier_id==s].critical,R[R.supplier_id==s].on_time),correction=False)[1] for s in S]
TC=pd.DataFrame([("on-time rate by order year (pooled)","chi-square",len(R),p_year),("monthly on-time rate vs month index (trend)","Spearman, permutation",len(mo),pm),("on-time rate, critical vs other POs (pooled)","chi-square",len(R),p_crit)],columns=["dimension","test","n","p_value"])
TC["p_holm"]=holm(TC.p_value.values); TC.to_csv(OUT/"w81_time_and_criticality.csv",index=False)
pd.DataFrame(dict(supplier_id=S,p_critical_vs_other=sup_crit,p_holm=holm(sup_crit))).to_csv(OUT/"w81_criticality_by_supplier.csv",index=False)
print(TC.round(4).to_string(index=False)); print("on-time: critical %.1f%% vs other %.1f%%"%(R[R.critical==1].on_time.mean()*100,R[R.critical==0].on_time.mean()*100))
print("years: "+", ".join(f"{y}: {v:.1f}%" for y,v in (R.groupby('year').on_time.mean()*100).items()))
check("six order years present",6,R.year.nunique(),R.year.nunique()==6)
sm=R.po_id.sample(200,random_state=SEED); bad=0
for po in sm:
    exp=int((L[L.po_id==po].criticality_level=="High").any()); bad+= (exp!=int(R.loc[R.po_id==po,"critical"].iloc[0]))
check("critical flag matches a plain filter on 200 sampled POs",0,bad,bad==0)

                                   dimension                  test     n  p_value  p_holm
         on-time rate by order year (pooled)            chi-square 21630   0.9545     1.0
 monthly on-time rate vs month index (trend) Spearman, permutation    72   0.8302     1.0
on-time rate, critical vs other POs (pooled)            chi-square 21630   0.5152     1.0
on-time: critical 84.9% vs other 85.5%
years: 2019: 84.5%, 2020: 85.1%, 2021: 85.1%, 2022: 85.2%, 2023: 84.7%, 2024: 84.9%


## 6. Concentration: how dependent is purchasing on any one supplier?
Measured on received spend and, for the chance test, on the number of PO lines per product. HHI is the sum of squared shares; with 8 suppliers its minimum is 0.125.

In [7]:
RL=L[L.po_status=="Received"]
sh=SC.spend_share.values/100; hhi=float((sh**2).sum()); print("supplier HHI (received spend): %.4f (minimum 0.125); largest share %.1f%%"%(hhi,sh.max()*100))
check("supplier HHI is at least 0.125 and below 0.13",0.125,round(hhi,4),0.125<=hhi<0.13)
check("HHI recomputed by loop",round(hhi,6),round(sum(x*x for x in sh),6),abs(hhi-sum(x*x for x in sh))<1e-12)
rg=SC.groupby("region").spend_share.sum().sort_values(ascending=False); ty=SC.groupby("supplier_type").spend_share.sum().sort_values(ascending=False)
print("spend share by region:",rg.round(1).to_dict()); print("spend share by type:",ty.round(1).to_dict())
pd.concat([rg.rename("spend_share_pct").to_frame().assign(group="region"),ty.rename("spend_share_pct").to_frame().assign(group="supplier_type")]).to_csv(OUT/"w81_region_type_concentration.csv")
# per product: HHI on received spend, and chance test on line counts
pv=RL.pivot_table(index="product_id",columns="supplier_id",values="line_total",aggfunc="sum",fill_value=0); pc=RL.pivot_table(index="product_id",columns="supplier_id",values="line_total",aggfunc="size",fill_value=0)
rows=[]; share_c=pc.div(pc.sum(axis=1),axis=0)
for pid in pv.index:
    sp=pv.loc[pid]/pv.loc[pid].sum(); n=int(pc.loc[pid].sum()); obs=float(((pc.loc[pid]/n)**2).sum())
    nullh=((rng.multinomial(n,[1/8]*8,size=2000)/n)**2).sum(axis=1); p=float((nullh>=obs).mean())
    rows.append(dict(product_id=pid,suppliers=int((pv.loc[pid]>0).sum()),lines=n,top_supplier=sp.idxmax(),top_share=float(sp.max()*100),hhi_spend=float((sp**2).sum()),hhi_lines=obs,p_chance=p))
PC=pd.DataFrame(rows); PC["p_holm"]=holm(PC.p_chance.values); PC.to_csv(OUT/"w81_product_concentration.csv",index=False)
check("thirty products",30,len(PC),len(PC)==30); check("every product is bought from all 8 suppliers",30,int((PC.suppliers==8).sum()),(PC.suppliers==8).all())
print("top supplier share per product: mean %.1f%%, max %.1f%%; products above chance before Holm: %d of 30 (expected about 1.5); after Holm: %d"%(PC.top_share.mean(),PC.top_share.max(),(PC.p_chance<.05).sum(),(PC.p_holm<.05).sum()))
check("no product is more concentrated than chance after Holm adjustment",0,int((PC.p_holm<.05).sum()),(PC.p_holm<.05).sum()==0)

supplier HHI (received spend): 0.1250 (minimum 0.125); largest share 12.9%
spend share by region: {'East China': 62.4, 'South China': 24.7, 'North China': 12.9}
spend share by type: {'OEM': 38.1, 'Distributor': 37.3, 'Local Vendor': 24.6}
top supplier share per product: mean 13.3%, max 13.9%; products above chance before Holm: 3 of 30 (expected about 1.5); after Holm: 0


## 7. Charts

In [8]:
fig,ax=plt.subplots(figsize=(8.4,4.8)); style(ax); y=np.arange(8)[::-1]; d=SC.sort_values("on_time_rate")
ax.errorbar(d.on_time_rate,np.arange(8),xerr=[d.on_time_rate-d.on_time_lo,d.on_time_hi-d.on_time_rate],fmt="o",color=BLUE,ecolor=INK,capsize=4,lw=1.3,ms=7)
ax.axvline(R.on_time.mean()*100,color=ORANGE,lw=1.5,ls="--"); ax.text(R.on_time.mean()*100+.1,7.45,"all suppliers %.1f%%"%(R.on_time.mean()*100),color=ORANGE,fontsize=9)
ax.set_yticks(np.arange(8)); ax.set_yticklabels(d.supplier_name,color=INK,fontsize=9); ax.yaxis.grid(False); ax.xaxis.grid(True,color=GRID,lw=.8)
ax.set_xlabel("Purchase orders received on or before the expected date, % (95% interval)",color=INK2); ax.set_title("All eight suppliers sit within the same on-time range",loc="left",color=INK,fontsize=12)
fig.tight_layout(); fig.savefig(CH/"w81_01_on_time.png",dpi=150,facecolor=SURF); plt.close(fig)
fig,ax=plt.subplots(figsize=(8.4,4.8)); style(ax); d=SC.sort_values("lead_mean"); x=np.arange(8)
ax.vlines(x,d.lead_p5,d.lead_p95,color=INK,lw=2); ax.scatter(x,d.lead_mean,color=BLUE,s=60,zorder=3,label="mean actual lead time"); ax.scatter(x,d.contract_lead_days,marker="_",color=ORANGE,s=400,zorder=4,label="contracted lead time")
ax.set_xticks(x); ax.set_xticklabels([n.replace(" Supplies","").replace(" Vendor","") for n in d.supplier_name],rotation=30,ha="right",color=INK,fontsize=9); ax.set_ylabel("Days from order to receipt (5th to 95th percentile)",color=INK2)
ax.legend(frameon=False,labelcolor=INK2,loc="upper left"); ax.set_title("Lead times differ between suppliers, but hardly vary within one",loc="left",color=INK,fontsize=12)
fig.tight_layout(); fig.savefig(CH/"w81_02_lead_time.png",dpi=150,facecolor=SURF); plt.close(fig)
fig,ax=plt.subplots(figsize=(8.4,4.8)); style(ax); d=PC.sort_values("top_share"); ax.bar(np.arange(30),d.top_share,color=BLUE,width=.7); ax.axhline(12.5,color=ORANGE,lw=1.5,ls="--"); ax.text(0,12.9,"even split 12.5%",color=ORANGE,fontsize=9)
ax.set_xticks(np.arange(30)); ax.set_xticklabels(d.product_id,rotation=90,color=INK,fontsize=7); ax.set_ylabel("Largest supplier's share of the product's spend, %",color=INK2); ax.set_ylim(0,max(20,d.top_share.max()+2))
ax.set_title("No product depends on one supplier",loc="left",color=INK,fontsize=12); fig.tight_layout(); fig.savefig(CH/"w81_03_concentration.png",dpi=150,facecolor=SURF); plt.close(fig)
check("three charts written","3",str(len(list(CH.glob('w81_0*.png')))),len(list(CH.glob('w81_0*.png')))==3)

## 8. Output files and validation

In [9]:
outs=sorted(p.name for p in OUT.glob("*.csv")); print(outs); check("nine result tables written","9",str(len(outs)),len(outs)==9)
V=pd.DataFrame(checks); V.insert(0,"Check_ID",range(1,len(V)+1)); V.to_csv("validation_report_week08_w81.csv",index=False)
print(len(V),"checks,",int((V.Status=="PASS").sum()),"pass"); assert (V.Status=="PASS").all()

['w81_criticality_by_supplier.csv', 'w81_lead_time_spread.csv', 'w81_on_time_by_year.csv', 'w81_product_concentration.csv', 'w81_region_type_concentration.csv', 'w81_reliability_score_check.csv', 'w81_supplier_scorecard.csv', 'w81_supplier_tests.csv', 'w81_time_and_criticality.csv']
29 checks, 29 pass
